# Campaign dataset: generation and checks

## Goal
Create clearly labeled synthetic daily advertising data for Hassan's Week 1 campaign analyzer. No real advertising records, people, or platform benchmarks are used.

## Setup
Requires Python 3, NumPy, and pandas. Run in a folder where you want the CSV. Rerunning overwrites `campaign_data_synthetic.csv` in that folder with the regenerated data.

### Key assumptions
One row is one campaign on one calendar date; all monetary values are USD; all conversions are attributed purchases, including Traffic and Awareness campaigns. A purchase is counted once against a single campaign in this simulation. Daily dates have no timezone conversion. Each campaign has one channel, audience, and goal. Revenue is known on all rows, excluding refunds, tax, shipping, and operating costs. Advertising-spend return is not full business profit ROI.


## Steps
### 1. Set dates and random seed

In [1]:
from datetime import date, timedelta
from decimal import Decimal
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 20261011
START_DATE = date(2026, 4, 1)
END_DATE = date(2026, 9, 30)
CSV_PATH = Path("campaign_data_synthetic.csv")
rng = np.random.default_rng(SEED)
print(f"Synthetic demonstration data; seed {SEED}; USD; {START_DATE} to {END_DATE}.")


Synthetic demonstration data; seed 20261011; USD; 2026-04-01 to 2026-09-30.


### 2. Generate campaigns and write the CSV
The simulation includes daily noise, weekend variation, a mid-period launch, campaign end dates, and explicitly simulated changes. Channel factors are chosen for demonstration; they do not describe real platform performance. Four documented rows exercise zero-denominator cases.

In [2]:
# Parameters are illustrative simulation choices, not advertising benchmarks.
channels = {
    "Facebook":  {"budget": 1.00, "cpm": 1.00, "ctr": 1.00, "cvr": 1.05},
    "Instagram": {"budget": 1.15, "cpm": 1.10, "ctr": 1.20, "cvr": 0.95},
    "Pinterest": {"budget": 0.72, "cpm": 0.80, "ctr": 0.80, "cvr": 1.10},
    "Twitter":   {"budget": 0.58, "cpm": 0.78, "ctr": 0.85, "cvr": 0.72},
}
# Campaign families: name, audience, goal, budget, CPM, CTR, purchase CVR, AOV, dates.
families = [
    ("Everyday Essentials", "Prospecting", "Sales", 110, 11, .012, .018, 105, START_DATE, END_DATE),
    ("Return to Cart", "Retargeting", "Sales", 75, 15, .021, .036, 110, START_DATE, END_DATE),
    ("Loyalty Collection", "Existing Customers", "Sales", 55, 13, .020, .030, 100, START_DATE, END_DATE),
    ("Summer Launch", "Prospecting", "Sales", 175, 12, .014, .016, 120, date(2026, 5, 15), date(2026, 8, 31)),
    ("Discover the Brand", "Prospecting", "Awareness", 70, 7.5, .006, .003, 95, date(2026, 4, 15), END_DATE),
    ("Explore the Store", "Prospecting", "Traffic", 65, 9, .023, .005, 100, START_DATE, END_DATE),
]
rows = []
campaign_number = 0
for channel, factors in channels.items():
    for family in families:
        campaign_number += 1
        name, audience, goal, budget, cpm, ctr, cvr, aov, first_day, last_day = family
        campaign_id = f"CMP{campaign_number:03d}"
        campaign_budget = budget * factors["budget"] * rng.uniform(.88, 1.12)
        campaign_quality = rng.uniform(.82, 1.18)
        current_day = first_day
        while current_day <= last_day:
            elapsed = (current_day - START_DATE).days
            weekend = current_day.weekday() >= 5
            seasonal = 1 + .10 * np.sin(2 * np.pi * elapsed / 31)
            spend = round(campaign_budget * seasonal * (1.14 if weekend else 1)
                          * rng.lognormal(-.015, .17), 2)
            daily_cpm = cpm * factors["cpm"] * rng.lognormal(0, .10)
            # Explicit simulated changes provide useful time-series demonstrations.
            if channel == "Twitter" and current_day >= date(2026, 8, 15):
                daily_cpm *= 1.22
            impressions = max(1, int(round(spend * 1000 / daily_cpm)))
            click_probability = ctr * factors["ctr"] * campaign_quality * rng.lognormal(0, .12)
            click_probability = min(.10, max(.0001, click_probability))
            clicks = int(rng.binomial(impressions, click_probability))
            purchase_probability = cvr * factors["cvr"] * campaign_quality
            purchase_probability *= 1.13 if weekend else 1
            if channel == "Instagram" and name == "Summer Launch" and current_day >= date(2026, 6, 15):
                purchase_probability *= 1.40
            if channel == "Pinterest" and goal == "Traffic" and current_day >= date(2026, 7, 1):
                purchase_probability *= 1.45
            if channel == "Twitter" and current_day >= date(2026, 8, 15):
                purchase_probability *= .72
            purchase_probability *= rng.lognormal(0, .16)
            conversions = int(rng.binomial(clicks, min(.25, max(0, purchase_probability))))
            revenue = round(conversions * aov * rng.uniform(.85, 1.15), 2) if conversions else 0.0
            rows.append({
                "date": current_day.isoformat(), "campaign_id": campaign_id,
                "campaign_name": f"{name} - {channel}", "channel": channel,
                "audience": audience, "goal": goal,
                "impressions": impressions, "clicks": clicks, "conversions": conversions,
                "spend": spend, "revenue": revenue, "currency": "USD", "dataset_type": "synthetic",
            })
            current_day += timedelta(days=1)

data = pd.DataFrame(rows).sort_values(["date", "campaign_id"]).reset_index(drop=True)
edge_cases = {
    ("2026-04-03", "CMP001"): {"impressions": 0, "clicks": 0, "conversions": 0, "spend": 0.0, "revenue": 0.0},
    ("2026-06-15", "CMP009"): {"impressions": 2050, "clicks": 0, "conversions": 0, "spend": 24.0, "revenue": 0.0},
    ("2026-05-09", "CMP014"): {"impressions": 1000, "clicks": 14, "conversions": 0, "spend": 7.8, "revenue": 0.0},
    ("2026-09-02", "CMP019"): {"impressions": 1000, "clicks": 10, "conversions": 2, "spend": 0.0, "revenue": 190.0},
}
for (day, campaign_id), values in edge_cases.items():
    selection = (data["date"] == day) & (data["campaign_id"] == campaign_id)
    assert selection.sum() == 1
    for column, value in values.items():
        data.loc[selection, column] = value
data.to_csv(CSV_PATH, index=False, encoding="utf-8", float_format="%.2f")
print(f"Wrote {len(data):,} daily records for {data.campaign_id.nunique()} campaigns.")


Wrote 4,040 daily records for 24 campaigns.


## Checks
### 3. Validate the saved data

In [3]:
# Validate the saved CSV, rather than only checking the generation dataframe.
loaded = pd.read_csv(CSV_PATH)
required = ["date", "campaign_id", "campaign_name", "channel", "audience", "goal",
            "impressions", "clicks", "conversions", "spend", "revenue", "currency", "dataset_type"]
assert list(loaded.columns) == required
assert len(loaded) == 4040
assert not loaded.isna().any().any()
assert not loaded.duplicated().any()
assert not loaded.duplicated(["date", "campaign_id", "channel"]).any()
dates = pd.to_datetime(loaded["date"], format="%Y-%m-%d", errors="raise")
assert dates.min().date() == START_DATE and dates.max().date() == END_DATE
assert dates.nunique() == 183
assert set(loaded["channel"]) == set(channels)
assert set(loaded["audience"]) == {"Prospecting", "Retargeting", "Existing Customers"}
assert set(loaded["goal"]) == {"Sales", "Traffic", "Awareness"}
assert loaded["currency"].eq("USD").all()
assert loaded["dataset_type"].eq("synthetic").all()
counts = ["impressions", "clicks", "conversions"]
assert all(pd.api.types.is_integer_dtype(loaded[c]) for c in counts)
assert loaded[counts + ["spend", "revenue"]].ge(0).all().all()
assert loaded["conversions"].le(loaded["clicks"]).all()
assert loaded["clicks"].le(loaded["impressions"]).all()
assert loaded.loc[loaded["conversions"].eq(0), "revenue"].eq(0).all()
assert loaded.loc[loaded["conversions"].gt(0), "revenue"].gt(0).all()
assert loaded.groupby("campaign_id")[["campaign_name", "channel", "audience", "goal"]].nunique().eq(1).all().all()
assert loaded.groupby("channel").size().eq(1010).all()
assert loaded.groupby("campaign_id")["date"].nunique().sort_values().tolist() == sorted([183, 183, 183, 109, 169, 183] * 4)
for (day, campaign_id), expected in edge_cases.items():
    actual = loaded.loc[(loaded.date == day) & (loaded.campaign_id == campaign_id)].iloc[0]
    assert all(actual[k] == v for k, v in expected.items())
print("Checks passed: dimensions, full date coverage, unique row keys, stable campaign metadata,")
print("nonnegative values, integer counts, funnel order, revenue consistency, and edge-case rows.")
print(loaded.head(3).to_string(index=False))


Checks passed: dimensions, full date coverage, unique row keys, stable campaign metadata,
nonnegative values, integer counts, funnel order, revenue consistency, and edge-case rows.
      date campaign_id                  campaign_name  channel           audience  goal  impressions  clicks  conversions  spend  revenue currency dataset_type
2026-04-01      CMP001 Everyday Essentials - Facebook Facebook        Prospecting Sales         9200      86            2  91.23   219.11      USD    synthetic
2026-04-01      CMP002      Return to Cart - Facebook Facebook        Retargeting Sales         4302      98            2  57.49   217.08      USD    synthetic
2026-04-01      CMP003  Loyalty Collection - Facebook Facebook Existing Customers Sales         6176     145            5  73.68   482.22      USD    synthetic


### 4. Reconcile totals and compute reference metrics
Aggregate raw numerators and denominators before calculating rates. Do not average daily CTR, CVR, CPC, CPA, ROAS, or returns.

In [4]:
# Independent currency totals use csv + Decimal to avoid binary rounding artifacts.
import csv
with CSV_PATH.open(newline="", encoding="utf-8") as handle:
    raw_rows = list(csv.DictReader(handle))
spend_total = sum((Decimal(r["spend"]) for r in raw_rows), Decimal("0"))
revenue_total = sum((Decimal(r["revenue"]) for r in raw_rows), Decimal("0"))
impression_total = sum(int(r["impressions"]) for r in raw_rows)
click_total = sum(int(r["clicks"]) for r in raw_rows)
conversion_total = sum(int(r["conversions"]) for r in raw_rows)
assert abs(Decimal(str(loaded.spend.sum())) - spend_total) < Decimal("0.005")
assert abs(Decimal(str(loaded.revenue.sum())) - revenue_total) < Decimal("0.005")
for field in counts:
    assert int(loaded[field].sum()) == sum(int(r[field]) for r in raw_rows)

def divide(numerator, denominator):
    return float(numerator) / float(denominator) if denominator else None

metrics = {
    "impressions": impression_total, "clicks": click_total, "conversions": conversion_total,
    "spend": str(spend_total), "revenue": str(revenue_total),
    "ctr": divide(click_total, impression_total), "cvr": divide(conversion_total, click_total),
    "cpc": divide(spend_total, click_total), "cpa": divide(spend_total, conversion_total),
    "roas": divide(revenue_total, spend_total), "ad_spend_return": divide(revenue_total - spend_total, spend_total),
}
print("All-data control totals (rates are fractions, not percentages):")
for name, value in metrics.items():
    print(f"{name}: {value}")
channel_controls = loaded.groupby("channel", sort=True)[counts + ["spend", "revenue"]].sum()
assert all(int(channel_controls[c].sum()) == metrics[c] for c in counts)
assert abs(float(channel_controls.spend.sum()) - float(spend_total)) < .005
assert abs(float(channel_controls.revenue.sum()) - float(revenue_total)) < .005
assert divide(0, 0) is None
print("Channel subtotals reconcile to global totals. Zero-denominator handling: N/A.")


All-data control totals (rates are fractions, not percentages):
impressions: 30156909
clicks: 437337
conversions: 7823
spend: 306470.04
revenue: 839690.13
ctr: 0.01450204992826022
cvr: 0.01788780734307868
cpc: 0.7007640332283799
cpa: 39.175513230218584
roas: 2.739876726612494
ad_spend_return: 1.7398767266124937
Channel subtotals reconcile to global totals. Zero-denominator handling: N/A.


### 5. Check combined filters and date boundaries

In [5]:
# Independent checks for a combined filter and inclusive date boundaries.
filtered = loaded.loc[(loaded.date >= "2026-06-01") & (loaded.date <= "2026-06-30")
                      & loaded.channel.eq("Instagram") & loaded.audience.eq("Prospecting")
                      & loaded.goal.eq("Sales")]
assert len(filtered) == 60  # two eligible campaigns x 30 days
selected_raw = [r for r in raw_rows if "2026-06-01" <= r["date"] <= "2026-06-30"
                and r["channel"] == "Instagram" and r["audience"] == "Prospecting" and r["goal"] == "Sales"]
assert filtered.campaign_id.nunique() == 2
assert len(selected_raw) == len(filtered)
assert int(filtered.conversions.sum()) == sum(int(r["conversions"]) for r in selected_raw)
assert loaded.loc[loaded.date.eq("2026-04-01")].shape[0] == 16
assert loaded.loc[loaded.date.eq("2026-09-30")].shape[0] == 20
assert loaded.loc[loaded.date.gt("2026-09-30")].empty
print("Filter check passed: June + Instagram + Prospecting + Sales = 60 rows across 2 campaigns.")
print("Boundary checks passed: April 1 has 16 rows; September 30 has 20; October has no rows.")


Filter check passed: June + Instagram + Prospecting + Sales = 60 rows across 2 campaigns.
Boundary checks passed: April 1 has 16 rows; September 30 has 20; October has no rows.


## Next Steps
Use the CSV as the app's default demonstration dataset and display a synthetic-data label. Check the app against these reference totals with all filters reset. Use the guide's documented rows to test N/A behavior. Missing revenue, malformed dates, and duplicate input rows are not included in this clean CSV; test those separately with edited copies.
